In [2]:
import os
print("Current Working Directory:", os.getcwd())
print("Files in this directory:", os.listdir())

Current Working Directory: C:\Users\inder\Firm Concentration project
Files in this directory: ['.ipynb_checkpoints', '01_market_analysis.ipynb', '02_policy_simulation.ipynb', 'market_data.db', 'market_data_db']


In [3]:
import sqlite3
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

#Connecting to SQL Database using in 01
conn = sqlite3.connect("market_data.db")

#Extracting transaction data
sql_baseline = """
SELECT
    transaction_id,
    quarter_date,
    region_id,
    firm_id,
    units_sold AS baseline_units,
    unit_cost,
    unit_price AS baseline_price,
    (units_sold * unit_price) AS baseline_revenue,
    (units_sold * (unit_price - unit_cost)) AS baseline_profit
FROM transactions;
"""

df_sim = pd.read_sql_query(sql_baseline, conn)

print(f"Success! Loaded {len(df_sim)} records")
df_sim.head()

Success! Loaded 160 records


,transaction_id,quarter_date,region_id,firm_id,baseline_units,unit_cost,baseline_price,baseline_revenue,baseline_profit
0,1,2024-Q1,North,Firm A,93,10.77,13.90,1292.70,291.09
1,2,2024-Q1,North,Firm B,188,10.06,13.27,2494.76,603.48
2,3,2024-Q1,North,Firm C,338,9.87,12.71,4295.98,959.92
3,4,2024-Q1,North,Firm D,376,9.88,12.76,4797.76,1082.88
4,5,2024-Q1,North,Firm E,404,8.87,11.25,4545.00,961.52


In [4]:
#Parameters for Policy
MARGIN_CAP = 0.25  #Maximum 25% margin over unit cost
PED = -0.75        #Inelastic Price Elasticity of Demand

#Calculating Price Ceiling
df_sim["max_allowed_price"] = df_sim["unit_cost"] * (1 + MARGIN_CAP)
df_sim["policy_price"] = np.minimum(df_sim["baseline_price"], df_sim["max_allowed_price"])

#Change in price (fractional)
df_sim["pct_change_price"] = (df_sim["policy_price"] - df_sim["baseline_price"]) / df_sim["baseline_price"]

#Demand adjustment using PED
df_sim["pct_change_units"] = PED * df_sim["pct_change_price"]
df_sim["policy_units"] = df_sim["baseline_units"] * (1 + df_sim["pct_change_units"])

#Post policy revenue and profit
df_sim["policy_revenue"] = df_sim["policy_price"] * df_sim["policy_units"]
df_sim["policy_profit"] = df_sim["policy_units"] * (df_sim["policy_price"] - df_sim["unit_cost"])

#No of affected transactions
affected_count = (df_sim["policy_price"] < df_sim["baseline_price"]).sum()

#No of firms affected
affected_firms = df_sim[df_sim["policy_price"] < df_sim["baseline_price"]]["firm_id"].nunique()

#Summaries
print(f"Policy in effect. The price cap bound on {affected_count} out of {len(df_sim)} transactions.")
print(f"Impacted {affected_firms} distinct firms.")

#Table preview
df_sim[["firm_id", "baseline_price", "policy_price", "baseline_units", "policy_units"]].head()

Policy in effect. The price cap bound on 97 out of 160 transactions.
Impacted 5 distinct firms.


,firm_id,baseline_price,policy_price,baseline_units,policy_units
0,Firm A,13.90,13.4625,93,95.195369
1,Firm B,13.27,12.5750,188,195.384702
2,Firm C,12.71,12.3375,338,345.429485
3,Firm D,12.76,12.3500,376,385.061129
4,Firm E,11.25,11.0875,404,408.376667


In [9]:
# Summing columns for market totals
total_baseline_revenue = df_sim["baseline_revenue"].sum()
total_policy_revenue = df_sim["policy_revenue"].sum()

total_baseline_profit = df_sim["baseline_profit"].sum()
total_policy_profit = df_sim["policy_profit"].sum()

total_baseline_units = df_sim["baseline_units"].sum()
total_policy_units = df_sim["policy_units"].sum()

#Market-wide percentage changes
pct_change_total_revenue = (total_policy_revenue - total_baseline_revenue) / total_baseline_revenue
pct_change_total_profit = (total_policy_profit - total_baseline_profit) / total_baseline_profit
pct_change_total_units = (total_policy_units - total_baseline_units) / total_baseline_units

#Formatted Market Summary 
print("--- MARKET IMPACT SUMMARY ---")
print(f"Total Volume Sold:  {total_baseline_units:,.0f} -> {total_policy_units:,.0f} units ({pct_change_total_units:+.2%})")
print(f"Total Revenue:      £{total_baseline_revenue:,.2f} -> £{total_policy_revenue:,.2f} ({pct_change_total_revenue:+.2%})")
print(f"Total Profit:       £{total_baseline_profit:,.2f} -> £{total_policy_profit:,.2f} ({pct_change_total_profit:+.2%})")

--- MARKET IMPACT SUMMARY ---
Total Volume Sold:  48,326 -> 49,194 units (+1.80%)
Total Revenue:      £781,854.11 -> £776,335.46 (-0.71%)
Total Profit:       £168,161.91 -> £151,706.23 (-9.79%)


In [12]:
conn = sqlite3.connect("market_data.db")
df_sim.to_sql("simulated_transactions", conn, if_exists="replace", index=False)
conn.close()
print('Success! Dataset saved to table named "simulated_transactions".')

Success! Dataset saved to table named "simulated_transactions".
